## GINN `best.pt` 推理与剖面对比

这个 notebook 会完成三件事：

1. 从 `experiments/ginn/checkpoints/best.pt` 加载训练好的模型。
2. 使用 `Trainer.predict_volume()` 对整个体做预测，得到阻抗体。
3. 在同一位置切出预测阻抗剖面和低频模型剖面，并排对比，同时给出差值剖面。


In [ ]:
import sys
from pathlib import Path

import cigsegy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_well_heads_petrel
from cup.seismic.survey import open_survey
from ginn.config import GINNConfig
from ginn.trainer import Trainer

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans SC", "SimSun"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
checkpoint_path = repo_root / "experiments" / "ginn" / "results" / "2026042301" / "checkpoints" / "best.pt"
output_dir = repo_root / "data" / "output_ginn_inversion_20260423"
output_dir.mkdir(parents=True, exist_ok=True)

slice_mode = "inline"  # 可选: "inline" 或 "xline"
slice_index = None  # None 表示默认取中间剖面
clip_percentiles = (1.0, 99.0)
ai_display_min = 0.0
ai_display_max = 20000.0

if not checkpoint_path.exists():
    raise FileNotFoundError(checkpoint_path)

checkpoint_path

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
cfg_payload = checkpoint["config"]
cfg = GINNConfig.from_dict(cfg_payload) if isinstance(cfg_payload, dict) else cfg_payload
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(cfg)
trainer.model.load_state_dict(checkpoint["model_state_dict"])
trainer.model.eval()

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Epoch: {checkpoint['epoch']}")
print(f"Best loss: {checkpoint['best_loss']:.6f}")
print(f"Device: {trainer.device}")
print(f"Geometry: {trainer.geometry}")


In [ ]:
pred_volume = trainer.predict_volume()

n_il = int(trainer.geometry["n_il"])
n_xl = int(trainer.geometry["n_xl"])
n_sample = int(trainer.geometry["n_sample"])

lfm_volume = trainer.dataset._lfm_flat.reshape(n_il, n_xl, n_sample)
mask_volume = trainer.dataset._mask_flat.reshape(n_il, n_xl, n_sample)

prediction_path = output_dir / "pred_volume_best.npy"
np.save(prediction_path, pred_volume.astype(np.float32))

print("Prediction volume shape:", pred_volume.shape)
print("LFM volume shape:", lfm_volume.shape)
print("Mask volume shape:", mask_volume.shape)
print("Saved prediction to:", prediction_path)


In [ ]:
def resolve_slice_index(mode: str, index: int | None, geometry: dict) -> int:
    if mode not in {"inline", "xline"}:
        raise ValueError(f"slice_mode must be 'inline' or 'xline', got {mode!r}")
    size = int(geometry["n_il"] if mode == "inline" else geometry["n_xl"])
    if index is None:
        return size // 2
    if not (0 <= index < size):
        raise IndexError(f"slice_index={index} out of range for {mode} size={size}")
    return index


def extract_section(volume: np.ndarray, mode: str, index: int) -> np.ndarray:
    if mode == "inline":
        return volume[index, :, :].T
    if mode == "xline":
        return volume[:, index, :].T
    raise ValueError(mode)


def extract_mask_section(mask: np.ndarray, mode: str, index: int) -> np.ndarray:
    if mode == "inline":
        return mask[index, :, :].T
    if mode == "xline":
        return mask[:, index, :].T
    raise ValueError(mode)


def robust_limits(*arrays: np.ndarray, percentiles=(1.0, 99.0)) -> tuple[float, float]:
    values = np.concatenate([np.asarray(arr, dtype=np.float32).ravel() for arr in arrays])
    return tuple(np.percentile(values, percentiles))


def sample_axis_seconds(geometry: dict) -> np.ndarray:
    n_sample = int(geometry["n_sample"])
    sample_min = float(geometry["sample_min"])
    sample_step = float(geometry["sample_step"])
    axis = sample_min + np.arange(n_sample, dtype=np.float64) * sample_step
    sample_unit = str(geometry.get("sample_unit", "")).strip().lower()
    if sample_unit in {"s", "sec", "secs", "second", "seconds"}:
        return axis
    if sample_unit in {"ms", "msec", "millisecond", "milliseconds"}:
        return axis / 1000.0
    if np.nanmax(np.abs(axis)) > 100.0:
        return axis / 1000.0
    return axis


def bilinear_trace_from_volume(volume: np.ndarray, survey_ctx, x: float, y: float) -> np.ndarray:
    i_float, j_float = survey_ctx.coord_to_index(x, y)
    i0 = int(np.floor(i_float))
    i1 = int(np.ceil(i_float))
    j0 = int(np.floor(j_float))
    j1 = int(np.ceil(j_float))
    i1 = min(i1, volume.shape[0] - 1)
    j1 = min(j1, volume.shape[1] - 1)
    wi = float(i_float - i0)
    wj = float(j_float - j0)
    t00 = volume[i0, j0, :]
    t01 = volume[i0, j1, :]
    t10 = volume[i1, j0, :]
    t11 = volume[i1, j1, :]
    return (1.0 - wi) * (1.0 - wj) * t00 + (1.0 - wi) * wj * t01 + wi * (1.0 - wj) * t10 + wi * wj * t11


def collect_wtie_references(ai_dir: Path) -> dict[str, Path]:
    refs: dict[str, Path] = {}
    for csv_path in sorted(Path(ai_dir).glob("*.csv")):
        stem = csv_path.stem
        if "_" not in stem:
            continue
        well_name, _ = stem.rsplit("_", 1)
        refs[well_name.upper()] = csv_path
    return refs


resolved_slice_index = resolve_slice_index(slice_mode, slice_index, trainer.geometry)
pred_section = extract_section(pred_volume, slice_mode, resolved_slice_index)
lfm_section = extract_section(lfm_volume, slice_mode, resolved_slice_index)
mask_section = extract_mask_section(mask_volume, slice_mode, resolved_slice_index)
diff_section = pred_section - lfm_section

auto_vmin, auto_vmax = robust_limits(pred_section, lfm_section, percentiles=clip_percentiles)
shared_vmin = auto_vmin if ai_display_min is None else float(ai_display_min)
shared_vmax = auto_vmax if ai_display_max is None else float(ai_display_max)
diff_abs = np.percentile(np.abs(diff_section), clip_percentiles[1])

print(f"Slice mode: {slice_mode}")
print(f"Slice index: {resolved_slice_index}")
print(f"Shared display range: [{shared_vmin:.2f}, {shared_vmax:.2f}]")
print(f"Diff display abs range: +/-{diff_abs:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), constrained_layout=True)

im0 = axes[0].imshow(pred_section, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax)
axes[0].set_title(f"Predicted AI | {slice_mode}={resolved_slice_index}")
axes[0].set_xlabel("Trace index")
axes[0].set_ylabel("Sample index")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(lfm_section, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax)
axes[1].set_title(f"LFM | {slice_mode}={resolved_slice_index}")
axes[1].set_xlabel("Trace index")
axes[1].set_ylabel("Sample index")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

im2 = axes[2].imshow(diff_section, cmap="seismic", aspect="auto", origin="upper", vmin=-diff_abs, vmax=diff_abs)
axes[2].set_title(f"Prediction - LFM | {slice_mode}={resolved_slice_index}")
axes[2].set_xlabel("Trace index")
axes[2].set_ylabel("Sample index")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

figure_path = output_dir / f"{slice_mode}_{resolved_slice_index:04d}_prediction_vs_lfm.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
print("Saved figure to:", figure_path)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
mask_image = np.where(mask_section, 1.0, 0.0)
im = ax.imshow(mask_image, cmap="gray_r", aspect="auto", origin="upper", vmin=0.0, vmax=1.0)
ax.set_title(f"Mask | {slice_mode}={resolved_slice_index}")
ax.set_xlabel("Trace index")
ax.set_ylabel("Sample index")
fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()


In [ ]:
orig_segy_path = Path(cfg.seismic_file)
if not orig_segy_path.exists():
    raise FileNotFoundError(orig_segy_path)

keylocs = [cfg.segy_iline, cfg.segy_xline, cfg.segy_istep, cfg.segy_xstep]
pred_segy_path = output_dir / "pred_volume_best.segy"

pred_volume_export = np.ascontiguousarray(pred_volume.astype(np.float32))


def build_textual_header(title: str, lines: list[str]) -> str:
    rows = [f"C{idx:>2d} {text}"[:80].ljust(80) for idx, text in enumerate([title, *lines], start=1)]
    rows.extend([f"C{idx:>2d}".ljust(80) for idx in range(len(rows) + 1, 41)])
    textual = "".join(rows)
    if len(textual) != 3200:
        raise ValueError(f"Expected 3200-char textual header, got {len(textual)}")
    return textual


pred_textual = build_textual_header(
    "GINN predicted impedance volume",
    [
        f"checkpoint={checkpoint_path.name}",
        f"epoch={checkpoint['epoch']}",
        f"best_loss={checkpoint['best_loss']:.6f}",
    ],
)

cigsegy.create_by_sharing_header(
    str(pred_segy_path),
    str(orig_segy_path),
    pred_volume_export,
    keylocs=keylocs,
    textual=pred_textual,
)

print("Exported predicted impedance SEG-Y to:", pred_segy_path)


In [ ]:
print("执行井验收...")
survey_ctx = open_survey(
    cfg.seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": cfg.segy_iline,
        "xline": cfg.segy_xline,
        "istep": cfg.segy_istep,
        "xstep": cfg.segy_xstep,
    },
)
sample_axis_s = sample_axis_seconds(survey_ctx.query_geometry(domain="time"))

well_heads = import_well_heads_petrel(repo_root / "data/raw/well_heads")
well_heads = well_heads.assign(Name_norm=well_heads["Name"].str.upper())
reference_files = collect_wtie_references(repo_root / "data/output_vertical_well_wtie_20260416/ai")

metrics = []
example_plot_done = False

for well_name_norm, csv_path in reference_files.items():
    head_match = well_heads.loc[well_heads["Name_norm"] == well_name_norm]
    if head_match.empty:
        print(f"跳过 {well_name_norm}: 井头未匹配")
        continue

    head = head_match.iloc[0]
    ref_df = pd.read_csv(csv_path)
    ref_df = ref_df.rename(columns={c: c.strip().lower() for c in ref_df.columns})
    if "twt_s" not in ref_df.columns or "ai" not in ref_df.columns:
        print(f"跳过 {well_name_norm}: 列名异常 {list(ref_df.columns)}")
        continue

    pred_trace = bilinear_trace_from_volume(pred_volume, survey_ctx, float(head["Surface X"]), float(head["Surface Y"]))
    pred_ai = np.interp(ref_df["twt_s"].to_numpy(), sample_axis_s, pred_trace, left=np.nan, right=np.nan)
    ref_ai = ref_df["ai"].to_numpy(dtype=np.float64)
    valid = np.isfinite(pred_ai) & np.isfinite(ref_ai)
    if not np.any(valid):
        print(f"跳过 {well_name_norm}: 无重叠采样")
        continue

    diff = pred_ai[valid] - ref_ai[valid]
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))
    bias = float(np.mean(diff))
    corr = float(np.corrcoef(pred_ai[valid], ref_ai[valid])[0, 1]) if np.sum(valid) > 1 else np.nan

    metrics.append(
        {
            "well_name": head["Name"],
            "n_samples": int(np.sum(valid)),
            "mae": mae,
            "rmse": rmse,
            "bias": bias,
            "corr": corr,
            "reference_file": csv_path.name,
        }
    )

    if not example_plot_done:
        fig, ax = plt.subplots(figsize=(5, 10))
        ax.plot(ref_ai[valid], ref_df.loc[valid, "twt_s"], label="WTIE AI", lw=2)
        ax.plot(pred_ai[valid], ref_df.loc[valid, "twt_s"], label="GINN 预测 AI", lw=2)
        ax.invert_yaxis()
        ax.set_xlabel("AI")
        ax.set_ylabel("TWT (s)")
        ax.set_title(f"井验收 | {head['Name']}")
        ax.legend(loc="best")
        ax.grid(True, alpha=0.3, linestyle=":")
        qc_plot_path = output_dir / f"well_qc_{str(head['Name']).replace('/', '_')}.png"
        fig.savefig(qc_plot_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("已保存井验收图:", qc_plot_path)
        example_plot_done = True

metrics_df = pd.DataFrame(metrics)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values("rmse")
metrics_csv_path = output_dir / "well_qc_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print("已保存井验收指标:", metrics_csv_path)
metrics_df
